# 第5章　代表的なモデルで「作る」 ― YOLO・nnU-Net・EfficientNet を使うだけ**『ゼロから動かす医療診断支援AI（入門編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-intro

## 5.2　検出をYOLOで ― 病変の位置を四角で示す

```text# Colabのセルで一度だけ!pip install ultralyticsfrom ultralytics import YOLO# ① 学習済みモデルを出発点にする（転移学習）model = YOLO("yolo11n.pt")# ② 自分のデータで学習（data=画像とラベルの場所を書いた設定ファイル）model.train(data="fracture.yaml", epochs=100, imgsz=640)# ③ 新しい画像で推論。conf=確信度の閾値（既定は0.25。医療では見逃さないよう下げる）results = model("new_xray.png", conf=0.10)results[0].show()     # 検出結果を枠つきで表示```

## 5.3　セグメンテーションをnnU-Netで ― 形まで塗り分ける

```bash# インストールpip install nnunetv2# ① データを検証し、前処理・設計を自動生成（ここが賢い）# データの置き場所を環境変数で教える（これを忘れると次のコマンドがエラーで止まる）# ※Colabでは各行を別セルの ! で実行すると export が持続しない。この一連を一つの %%bash セルにまとめるか、os.environ["nnUNet_raw"]=... で設定するexport nnUNet_raw="/content/nnUNet_raw"export nnUNet_preprocessed="/content/nnUNet_preprocessed"export nnUNet_results="/content/nnUNet_results"nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity# ② 学習（5分割のうちの一つ目を学習）# ※既定の学習設定を最後まで回すには非常に長い時間がかかり、無料Colabのセッションでは完走できない。#   入門編では①の前処理（plan_and_preprocess）までを体験し、本格的な学習は腰を据えた環境で回すのが現実的。nnUNetv2_train 1 3d_fullres 0# ③ 新しいCT/画像に予測を走らせるnnUNetv2_predict -i 入力フォルダ -o 出力フォルダ -d 1 -c 3d_fullres -f 0   # 学習したfoldを明示する```

## 5.4　分類をEfficientNetで ― 良悪性・疾患名を診断する

```text!pip install timmimport timm, torch# ① 学習済みEfficientNetを出発点に、自分のクラス数（例：良性/悪性=2）に付け替えるmodel = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2)# ② あとは学習ループを回すだけ（詳しい書き方は基礎編の実装ガイドの章で。まずは"こう書く"を眺める）optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)criterion = torch.nn.CrossEntropyLoss()# for images, labels in loader: ... （順伝播→誤差→逆伝播→更新）```

## 学習済みモデルで、まず1枚を推論してみる ― 「確率が出る」感覚をつかむ最小例

In [ ]:
import torchfrom torchvision.models import resnet18, ResNet18_Weightsfrom PIL import Imageweights = ResNet18_Weights.DEFAULTmodel = resnet18(weights=weights).eval()   # ImageNet学習済み。推論だけなので eval()preprocess = weights.transforms()          # 学習時と“同じ”前処理（リサイズ・正規化）が付いてくるimg = Image.open("sample.jpg").convert("RGB")   # 手元の1枚（猫でも犬でも何でもよい）x = preprocess(img).unsqueeze(0)                # (3,H,W) → (1,3,H,W) にバッチ次元を足すwith torch.no_grad():                           # 推論では勾配を計算しない（省メモリ・高速）    probs = model(x).softmax(dim=1)[0]          # ロジット → softmax で確率へtop5 = probs.topk(5)                            # 確率の高い上位5クラスfor p, i in zip(top5.values, top5.indices):    print(f"{weights.meta['categories'][i]:22s} {p.item():.3f}")